To run any individual cell, press shift+enter. The first cell installs pyspark, so click on the cell below and hit shift+enter to install spark into this environment

In [25]:
!pip install pyspark

In [26]:
!pip install pandas

In [27]:
!pip install numpy

In [28]:
import numpy as np
import pandas as pd

In [29]:
# Import SparkSession
from pyspark.sql import SparkSession
# Create a Spark Session
spark = SparkSession.builder.appName("excercise").getOrCreate()
# Check Spark Session Information
spark

#spark.stop()

In [30]:
#load data
df = spark.read.csv("IRIS.csv", header=True, inferSchema=True)
df.show()

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| Setosa|
|         4.9|        3.0|         1.4|        0.2| Setosa|
|         4.7|        3.2|         1.3|        0.2| Setosa|
|         4.6|        3.1|         1.5|        0.2| Setosa|
|         5.0|        3.6|         1.4|        0.2| Setosa|
|         5.4|        3.9|         1.7|        0.4| Setosa|
|         4.6|        3.4|         1.4|        0.3| Setosa|
|         5.0|        3.4|         1.5|        0.2| Setosa|
|         4.4|        2.9|         1.4|        0.2| Setosa|
|         4.9|        3.1|         1.5|        0.1| Setosa|
|         5.4|        3.7|         1.5|        0.2| Setosa|
|         4.8|        3.4|         1.6|        0.2| Setosa|
|         4.8|        3.0|         1.4|        0.1| Setosa|
|         4.3|        3.0|         1.1| 

In [31]:
#Handling categorical data
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCol="species", outputCol="species_indexed")
#fit to the data
df_indexed = indexer.fit(df).transform(df)

df_indexed.show(5)

+------------+-----------+------------+-----------+-------+---------------+
|sepal_length|sepal_width|petal_length|petal_width|species|species_indexed|
+------------+-----------+------------+-----------+-------+---------------+
|         5.1|        3.5|         1.4|        0.2| Setosa|            0.0|
|         4.9|        3.0|         1.4|        0.2| Setosa|            0.0|
|         4.7|        3.2|         1.3|        0.2| Setosa|            0.0|
|         4.6|        3.1|         1.5|        0.2| Setosa|            0.0|
|         5.0|        3.6|         1.4|        0.2| Setosa|            0.0|
+------------+-----------+------------+-----------+-------+---------------+
only showing top 5 rows


In [32]:
#Assemble independent features and Label

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
    outputCol="features")

output = assembler.transform(df_indexed)
output.show(5)

+------------+-----------+------------+-----------+-------+---------------+-----------------+
|sepal_length|sepal_width|petal_length|petal_width|species|species_indexed|         features|
+------------+-----------+------------+-----------+-------+---------------+-----------------+
|         5.1|        3.5|         1.4|        0.2| Setosa|            0.0|[5.1,3.5,1.4,0.2]|
|         4.9|        3.0|         1.4|        0.2| Setosa|            0.0|[4.9,3.0,1.4,0.2]|
|         4.7|        3.2|         1.3|        0.2| Setosa|            0.0|[4.7,3.2,1.3,0.2]|
|         4.6|        3.1|         1.5|        0.2| Setosa|            0.0|[4.6,3.1,1.5,0.2]|
|         5.0|        3.6|         1.4|        0.2| Setosa|            0.0|[5.0,3.6,1.4,0.2]|
+------------+-----------+------------+-----------+-------+---------------+-----------------+
only showing top 5 rows


In [33]:
#Finalized Dataset
finalized_df = output.select(["features", "species_indexed"])
finalized_df.show(5)

+-----------------+---------------+
|         features|species_indexed|
+-----------------+---------------+
|[5.1,3.5,1.4,0.2]|            0.0|
|[4.9,3.0,1.4,0.2]|            0.0|
|[4.7,3.2,1.3,0.2]|            0.0|
|[4.6,3.1,1.5,0.2]|            0.0|
|[5.0,3.6,1.4,0.2]|            0.0|
+-----------------+---------------+
only showing top 5 rows


In [34]:
#Split the data
train_data, test_data = finalized_df.randomSplit([0.75, 0.25], seed=42)

In [35]:
#Modeling
from pyspark.ml.classification import RandomForestClassifier

#instantiate model
rfc = RandomForestClassifier(featuresCol="features", labelCol="species_indexed", seed=42)

#fit the model
rfc_model = rfc.fit(train_data)

In [36]:
preds = rfc_model.transform(test_data)
preds.show(5)

+-----------------+---------------+--------------+-------------+----------+
|         features|species_indexed| rawPrediction|  probability|prediction|
+-----------------+---------------+--------------+-------------+----------+
|[4.4,3.0,1.3,0.2]|            0.0|[20.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[4.6,3.2,1.4,0.2]|            0.0|[20.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[4.6,3.6,1.0,0.2]|            0.0|[20.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[4.7,3.2,1.3,0.2]|            0.0|[20.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|[4.8,3.1,1.6,0.2]|            0.0|[20.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
+-----------------+---------------+--------------+-------------+----------+
only showing top 5 rows


In [37]:
#Evaluation
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator =  MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="species_indexed")

accuracy = evaluator.evaluate(preds)
print(f"Accuracy: {accuracy}")


Accuracy: 0.9707530070851871


In [38]:
#select and convert your model to RDD floats
confusion_matrix = preds.groupBy("species_indexed").pivot("prediction").count().na.fill(0)

confusion_matrix.orderBy("species_indexed").show()

+---------------+---+---+---+
|species_indexed|0.0|1.0|2.0|
+---------------+---+---+---+
|            0.0| 15|  0|  0|
|            1.0|  0| 10|  1|
|            2.0|  0|  0|  8|
+---------------+---+---+---+

